# World Cup 2026 - Prediction Model Analysis & Notebook

This notebook provides a comprehensive walkthrough of the World Cup 2026 prediction model, including:
- Historical data analysis
- Model training
- Tournament predictions
- Top goalscorer predictions
- Golden Ball predictions

**Current Status**: June 19, 2026 - Mexico has qualified to Round of 32

## 1. Setup and Imports

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Import our modules
from src.data_loader import DataLoader
from src.tournament_predictor import TournamentPredictor
from src.goalscorer_predictor import GoalscorerPredictor
from src.golden_ball_predictor import GoldenBallPredictor
from src.model import MatchOutcomeModel, TournamentSimulator
from src.utils import TeamStats, PlayerStats

print("✓ All modules imported successfully")
print(f"Current Date: {datetime.now().strftime('%B %d, %Y')}")

## 2. Load Data

In [ ]:
# Load all data
loader = DataLoader()
data = loader.get_all_data()

historical_matches = data['historical_matches']
player_stats = data['player_stats']
wc_2026_teams = data['wc_2026_teams']

print("Loaded Data:")
print(f"  - Historical Matches: {len(historical_matches)} records")
print(f"  - Player Statistics: {len(player_stats)} records")
print(f"  - WC 2026 Teams: {len(wc_2026_teams)} teams")

# Display sample data
print("\nSample Historical Matches:")
print(historical_matches.head())

## 3. Historical Data Analysis

In [ ]:
# Analyze historical matches
print("Historical Match Analysis:")
print(f"Total matches: {len(historical_matches)}")
print(f"Total tournaments: {historical_matches['tournament'].nunique()}")
print(f"\nTournaments represented:")
print(historical_matches['tournament'].value_counts())

# Goals statistics
total_goals = historical_matches['home_goals'].sum() + historical_matches['away_goals'].sum()
avg_goals_per_match = total_goals / len(historical_matches)
print(f"\nGoals Statistics:")
print(f"Total goals: {total_goals}")
print(f"Average goals per match: {avg_goals_per_match:.2f}")

# Win/Draw/Loss distribution
home_wins = len(historical_matches[historical_matches['home_goals'] > historical_matches['away_goals']])
draws = len(historical_matches[historical_matches['home_goals'] == historical_matches['away_goals']])
away_wins = len(historical_matches[historical_matches['home_goals'] < historical_matches['away_goals']])

print(f"\nMatch Results Distribution:")
print(f"Home wins: {home_wins} ({100*home_wins/len(historical_matches):.1f}%)")
print(f"Draws: {draws} ({100*draws/len(historical_matches):.1f}%)")
print(f"Away wins: {away_wins} ({100*away_wins/len(historical_matches):.1f}%)")

## 4. Team Analysis

In [ ]:
# Analyze World Cup 2026 teams
print("World Cup 2026 - Participating Teams")
print("\nTeams by FIFA Ranking:")

wc_sorted = wc_2026_teams.sort_values('fifa_rank')
print(wc_sorted[['team', 'fifa_rank', 'group']].to_string(index=False))

# Teams by group
print("\n" + "="*50)
print("Teams by Group:")
for group in sorted(loader.groups.keys()):
    teams = loader.groups[group]
    print(f"\nGroup {group}: {', '.join(teams)}")

## 5. Player Analysis

In [ ]:
# Top players analysis
print("Top 10 Players by Rating:")
top_players = player_stats.nlargest(10, 'rating')[['player_name', 'team', 'position', 'rating', 'goals', 'games_played']]
top_players['goals_per_game'] = top_players['goals'] / top_players['games_played']
print(top_players.to_string(index=False))

# Top scorers
print("\n" + "="*60)
print("Top 10 Scorers (by total goals):")
top_scorers = player_stats.nlargest(10, 'goals')[['player_name', 'team', 'goals', 'games_played']]
top_scorers['goals_per_game'] = top_scorers['goals'] / top_scorers['games_played']
print(top_scorers.to_string(index=False))

## 6. Train Machine Learning Model

In [ ]:
# Train the match outcome prediction model
print("Training Machine Learning Model...\n")
ml_model = MatchOutcomeModel()
model, accuracy = ml_model.train()

print("\n✓ Model training complete!")

## 7. Tournament Predictions

In [ ]:
# Run tournament predictions
tournament_predictor = TournamentPredictor()
champion = tournament_predictor.run_full_tournament_simulation()

## 8. Top Goalscorer Prediction

In [ ]:
# Predict top scorers
goalscorer_predictor = GoalscorerPredictor()
top_scorers_df = goalscorer_predictor.predict_top_scorers(top_n=10)

# Display as dataframe
print("\nTop Goalscorers (DataFrame View):")
print(top_scorers_df[['player', 'team', 'predicted_goals', 'scoring_potential']].to_string())

## 9. Golden Ball Prediction

In [ ]:
# Predict Golden Ball winner
golden_ball_predictor = GoldenBallPredictor()
gb_candidates = golden_ball_predictor.predict_winner(top_n=5)

# Display as dataframe
print("\nGolden Ball Candidates (DataFrame View):")
print(gb_candidates[['player', 'team', 'rating', 'final_score']].to_string())

## 10. Key Predictions Summary

In [ ]:
print("\n" + "#"*70)
print("WORLD CUP 2026 - FINAL PREDICTIONS".center(70))
print("#"*70)

print(f"\n🏆 TOURNAMENT CHAMPION: {champion if champion else 'TBD'}")

if len(top_scorers_df) > 0:
    top_scorer = top_scorers_df.iloc[0]
    print(f"⚽ TOP GOALSCORER: {top_scorer['player']} ({top_scorer['team']})")
    print(f"   Predicted Goals: {top_scorer['predicted_goals']}")

if len(gb_candidates) > 0:
    gb_winner = gb_candidates.iloc[0]
    print(f"🌟 GOLDEN BALL: {gb_winner['player']} ({gb_winner['team']})")
    print(f"   Rating: {gb_winner['rating']:.1f}/10")
    print(f"   Score: {gb_winner['final_score']:.2f}")

print(f"\n📊 Model Accuracy: {accuracy if accuracy else 'N/A'}")
print("\n" + "#"*70 + "\n")

## 11. Visualization (Optional)

In [ ]:
# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Plot 1: Top Players by Rating
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Top rated players
top_10_rated = player_stats.nlargest(10, 'rating')
axes[0, 0].barh(top_10_rated['player_name'], top_10_rated['rating'], color='steelblue')
axes[0, 0].set_xlabel('Rating')
axes[0, 0].set_title('Top 10 Players by Rating')
axes[0, 0].invert_yaxis()

# Top scorers
if len(top_scorers_df) > 0:
    top_5_scorers = top_scorers_df.head(5)
    axes[0, 1].bar(range(len(top_5_scorers)), top_5_scorers['predicted_goals'].values, color='coral')
    axes[0, 1].set_xticks(range(len(top_5_scorers)))
    axes[0, 1].set_xticklabels([p.split()[0] for p in top_5_scorers['player'].values], rotation=45)
    axes[0, 1].set_ylabel('Predicted Goals')
    axes[0, 1].set_title('Top 5 Predicted Goalscorers')

# Teams by FIFA Ranking
top_teams = wc_2026_teams.nsmallest(10, 'fifa_rank')
axes[1, 0].barh(top_teams['team'], top_teams['fifa_rank'], color='lightgreen')
axes[1, 0].set_xlabel('FIFA Ranking (Lower is Better)')
axes[1, 0].set_title('Top 10 Teams by FIFA Ranking')
axes[1, 0].invert_xaxis()
axes[1, 0].invert_yaxis()

# Historical goals distribution
all_goals = list(historical_matches['home_goals'].values) + list(historical_matches['away_goals'].values)
axes[1, 1].hist(all_goals, bins=range(0, max(all_goals)+2), color='purple', alpha=0.7, edgecolor='black')
axes[1, 1].set_xlabel('Goals per Match')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Historical Goals Distribution')

plt.tight_layout()
plt.savefig('predictions_summary.png', dpi=150, bbox_inches='tight')
print("✓ Visualizations saved to 'predictions_summary.png'")
plt.show()

## 12. Model Performance and Next Steps

### Key Insights:
- The model is trained on historical World Cup data (2014, 2018, 2022)
- Multiple ML algorithms are used for prediction (Random Forest, Gradient Boosting, Logistic Regression)
- Predictions account for team strength, player form, and tournament structure

### How to Use:
1. Run `python main.py` for full predictions
2. Or use individual modules:
   - `tournament_predictor.py` for tournament outcomes
   - `goalscorer_predictor.py` for top scorers
   - `golden_ball_predictor.py` for best player

### Future Enhancements:
- Integrate real-time match data as tournament progresses
- Add more granular player statistics
- Incorporate betting odds for calibration
- Add injury/suspension updates
- Implement ensemble voting from multiple models